# pdf_vlm — Document QA experiment (Colab)

## What this experiment measures

Length-truncated packs (**5 / 10 / 20 / 50 / 100** pages) from **200+ page** reports:

| Axis | Variants |
|---|---|
| OCR | PP-StructureV3 (**tables ON** on Colab) + PDF text enrich |
| RAG generation | **text-only** vs **multimodal** (page images; shared text retriever) |
| Retrieval | **page-level** vs **hierarchical** |
| Metric | ANLS / EM / F1 + recall@k (needs Gemma GGUF for real answers) |

**Runtime:** GPU (T4+) recommended.

> Do **not** clone into `/content/pdf_vlm` — that folder name shadows the Python package.
>
> After code updates: re-run §0 (`git pull`), then **§4b** for multimodal re-eval (Korean, no `[page N]`).

## 0. Clone + install (package + OCR + llama.cpp)

In [1]:
import sys, shutil
from pathlib import Path

REPO_URL = "https://github.com/mAn-He/pdf_vlm.git"
ROOT = Path("/content/pdf_vlm_repo")

# Remove shadowed clone path if present
shadow = Path("/content/pdf_vlm")
if shadow.exists() and shadow.resolve() != ROOT.resolve():
    shutil.rmtree(shadow, ignore_errors=True)

if not (ROOT / "pyproject.toml").exists():
    !git clone --depth 1 {REPO_URL} {ROOT}
else:
    print("Repo present:", ROOT)
    # Pull latest prompts / harness fixes (needed for §4b Korean + no-citation MM)
    %cd {ROOT}
    !git pull --ff-only origin main || true

%cd {ROOT}
for k in list(sys.modules):
    if k == "pdf_vlm" or k.startswith("pdf_vlm."):
        del sys.modules[k]
sys.path.insert(0, str(ROOT / "src"))
print("cwd:", Path.cwd())

Cloning into '/content/pdf_vlm_repo'...
remote: Enumerating objects: 213, done.
remote: Counting objects: 100% (213/213), done.
remote: Compressing objects: 100% (179/179), done.
remote: Total 213 (delta 32), reused 194 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (213/213), 1.18 MiB | 16.79 MiB/s, done.
Resolving deltas: 100% (32/32), done.
/content/pdf_vlm_repo
cwd: /content/pdf_vlm_repo


In [ ]:
import subprocess, sys
from pathlib import Path

ROOT = Path("/content/pdf_vlm_repo").resolve()
assert (ROOT / "src/pdf_vlm/utils/io.py").exists()

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"])
# index + OCR (tables) + viz
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[index,ocr,viz]"])

def try_install_llama():
    for url in [
        "https://abetlen.github.io/llama-cpp-python/whl/cu124",
        "https://abetlen.github.io/llama-cpp-python/whl/cu122",
        None,
    ]:
        cmd = [sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python"]
        if url:
            cmd += ["--extra-index-url", url]
        print("Trying llama-cpp:", url or "default")
        if subprocess.run(cmd).returncode == 0:
            return True
    return False

print("llama-cpp:", try_install_llama())

for k in list(sys.modules):
    if k == "pdf_vlm" or k.startswith("pdf_vlm."):
        del sys.modules[k]
sys.path.insert(0, str(ROOT / "src"))

import pdf_vlm
from pdf_vlm.utils.io import project_root
from pdf_vlm.ocr.paddle_structure import paddle_available
print("pdf_vlm:", pdf_vlm.__file__)
print("paddle:", paddle_available())

import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

Trying llama-cpp: https://abetlen.github.io/llama-cpp-python/whl/cu124
Trying llama-cpp: https://abetlen.github.io/llama-cpp-python/whl/cu124


## 1. Download Gemma GGUF (required for real QA answers)

1. Accept: https://huggingface.co/google/gemma-3-4b-it-qat-q4_0-gguf  
2. Colab secret `HF_TOKEN` or paste token  
3. Set `DOWNLOAD_GGUF = True` below

If GGUF is missing, the harness can still measure **retrieval**, but **ANLS/QA quality will be empty** (dry-run).

In [ ]:
from pathlib import Path
import os

DOWNLOAD_GGUF = True  # set False only if you intentionally skip generation

gguf = Path("models/gemma-3-4b-it-q4_0.gguf")
mmproj = Path("models/mmproj-model-f16-4B.gguf")

if DOWNLOAD_GGUF and not (gguf.exists() and mmproj.exists()):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        from getpass import getpass
        os.environ["HF_TOKEN"] = getpass("HF token: ")
    !huggingface-cli login --token "$HF_TOKEN" --add-to-git-credential
    !{sys.executable} scripts/download_models.py --with-mmproj

print("gguf:", gguf.exists(), gguf)
print("mmproj:", mmproj.exists(), mmproj)
HAS_GGUF = gguf.exists() and mmproj.exists()
print("HAS_GGUF:", HAS_GGUF)

## 2. Build length packs 5/10/20/50/100 (optional if already in repo)

Repo already ships truncated PDFs under `data/custom/{5,10,20,50,100}/`.  
If you uploaded full `QA_report_HW.pdf` to the repo root, you can rebuild packs here.

In [ ]:
from pathlib import Path
import sys

BUCKETS = "5,10,20,50,100"
src = Path("QA_report_HW.pdf")

if src.exists():
    !{sys.executable} scripts/prepare_hw_report_dataset.py --pdf {src} --buckets {BUCKETS}
else:
    print("No QA_report_HW.pdf in repo root — using existing data/custom packs.")

for b in [5, 10, 20, 50, 100]:
    d = Path(f"data/custom/{b}")
    pdfs = list(d.glob("*.pdf")) if d.exists() else []
    print(f"bucket={b}: pdfs={[p.name for p in pdfs]} q={(d/'questions.json').exists()}")

## 3. OCR (PP-StructureV3, tables ON) + build retrieval indexes

Uses `configs/ocr/pp_structure_v3_colab.yaml` (`use_table_recognition: true`).  
This is the step that actually pulls OCR/table text used by RAG.

In [ ]:
import sys

BUCKETS = "5,10,20,50,100"
# Start smaller if Colab RAM is tight: BUCKETS = "5,10,20"

!{sys.executable} scripts/colab_prepare_custom.py \
  --buckets {BUCKETS} \
  --no-stub \
  --enrich-pdf-text \
  --hash-embedder \
  --ocr-config ocr/pp_structure_v3_colab.yaml \
  --force

from pdf_vlm.utils.io import load_json, resolve_path
prep = load_json(resolve_path("data/custom/colab_prepared.json"))
print(prep)
assert prep.get("items"), "No docs prepared — check data/custom manifests/PDFs"

## 4. Run eval matrix (RAG variants × page lengths)

Cells: **text/multimodal × page/hierarchical × custom_{5,10,20,50,100}**  
After this full matrix, use **§4b** to re-run multimodal only with Korean / no-citation prompts (requires latest `prompt_builder.py`).  
If `HAS_GGUF` is False → forced dry-run (retrieval only, ANLS≈0).

In [ ]:
import json, subprocess, sys, gc
from pathlib import Path

# Sync questions doc_id -> existing Colab indices (fixes Windows hash mismatch)
id_map = {}
for d in Path("indices").iterdir():
    if d.is_dir() and (d / "page_text").exists():
        id_map[d.name.rsplit("_", 1)[0]] = d.name
print("index stems:", id_map)

for n in [5, 10, 20, 50, 100]:
    key = f"hyundai_wia_qa_report_{n}p"
    new_id = id_map.get(key)
    if not new_id:
        print("NO INDEX for", key); continue
    qpath = Path(f"data/custom/{n}/questions.json")
    if qpath.exists():
        rows = json.loads(qpath.read_text(encoding="utf-8"))
        for r in rows:
            r["doc_id"] = new_id
        qpath.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    mpath = Path(f"data/custom/{n}/manifest.json")
    if mpath.exists():
        man = json.loads(mpath.read_text(encoding="utf-8"))
        for d in man.get("documents") or []:
            d["doc_id"] = new_id
        mpath.write_text(json.dumps(man, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"synced {n} -> {new_id}")

try:
    import torch
    torch.cuda.empty_cache()
except Exception:
    pass

def run(pipelines, datasets):
    cmd = [
        sys.executable, "scripts/run_eval_harness.py",
        "--config", "configs/experiments/eval_hw_wia_colab.yaml",
        "--datasets", datasets,
        "--pipelines", pipelines,
        "--retrievals", "page,hierarchical",
        "--top-k", "3",
        "--device", "cuda",
    ]
    print(">>>", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print("exit:", r.returncode)
    print(r.stdout[-4000:])
    if r.returncode != 0:
        print(r.stderr[-4000:])
    gc.collect()
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass
    return r.returncode

for ds in ["custom_5", "custom_10", "custom_20", "custom_50", "custom_100"]:
    assert run("text", ds) == 0, ds
    assert run("multimodal", ds) == 0, ds

In [ ]:
from pathlib import Path
import json

runs = sorted(Path("results/runs").glob("eval_*"), key=lambda p: p.stat().st_mtime, reverse=True)
print("latest:", runs[0] if runs else None)
if runs:
    for name in ["summary.json", "report.md", "aggregates.json"]:
        p = runs[0] / name
        if p.exists():
            print("====", name, "====")
            txt = p.read_text(encoding="utf-8")
            print(txt[:5000])
            break

## 4b. Multimodal-only re-eval (Korean + no page citations)

**When to run:** after pulling a repo revision that updates `src/pdf_vlm/rag/prompt_builder.py` (no `[page N]`, answer in Korean).

This cell re-runs **multimodal only** so you can compare raw ANLS against the old citation-forcing prompts — without redoing text RAG or OCR.

Prereqs: indexes already built (section 3), GGUF downloaded (section 1), questions synced (section 4).

In [ ]:
# Multimodal-only re-eval with updated prompts (Korean, no citations)
import gc
import importlib
import subprocess
import sys
from pathlib import Path

# Confirm prompt fix is loaded (must forbid citations, ask for Korean)
import pdf_vlm.rag.prompt_builder as pb
importlib.reload(pb)
sp = pb.SYSTEM_PROMPT_MULTIMODAL.lower()
assert "korean" in sp
assert "no page citations" in sp or "do not write [page" in sp
assert "always mention the supporting page" not in sp
assert "cite the supporting page" not in pb.MULTIMODAL_USER_TEMPLATE.lower()
print("OK: multimodal prompts forbid page citations; ask for Korean when Q is Korean")
print(pb.SYSTEM_PROMPT_MULTIMODAL)

gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass

DATASETS = "custom_5,custom_10,custom_20,custom_50,custom_100"  # or "custom_5" for a smoke pass
cmd = [
    sys.executable,
    "scripts/run_eval_harness.py",
    "--config",
    "configs/experiments/eval_hw_wia_colab.yaml",
    "--datasets",
    DATASETS,
    "--pipelines",
    "multimodal",
    "--retrievals",
    "page,hierarchical",
    "--top-k",
    "3",
]
print("Running:", " ".join(cmd))
subprocess.check_call(cmd)

In [ ]:
# Summarize latest multimodal runs (raw ANLS) + offline citation-strip on newest preds
import csv
import json
import re
from pathlib import Path

from pdf_vlm.eval.anls import anls

CITE = re.compile(r"\s*\[(?:pages?|Pages?|page|Page)\s*[^\]]*\]\s*", re.I)

def strip_cites(s: str) -> str:
    return re.sub(r"\s+", " ", CITE.sub(" ", s or "")).strip()

def golds(row):
    raw = row.get("gold_answers") or row.get("gold_answer") or ""
    if "|" in raw:
        return [x.strip() for x in raw.split("|") if x.strip()]
    return [raw.strip()] if raw.strip() else []

runs = sorted(Path("results/runs").glob("eval_*"), key=lambda p: p.stat().st_mtime)
mm_runs = []
for run in runs:
    hr = run / "harness_result.json"
    if not hr.exists():
        continue
    j = json.loads(hr.read_text(encoding="utf-8"))
    if (j.get("n_rows") or 0) <= 0:
        continue
    pred = run / "predictions.csv"
    if not pred.exists():
        continue
    with pred.open(encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    if not rows or rows[0].get("pipeline_type") != "multimodal":
        # mixed runs: keep only if majority multimodal
        mm = [r for r in rows if r.get("pipeline_type") == "multimodal"]
        if len(mm) < max(1, len(rows) // 2):
            continue
        rows = mm
    ov = j.get("overall") or {}
    mm_runs.append((run.name, rows, ov.get("correctness_mean") or ov.get("anls_mean")))

if not mm_runs:
    print("No multimodal runs with predictions found.")
else:
    name, rows, overall = mm_runs[-1]
    raw_vals, strip_vals = [], []
    eng = 0
    for r in rows:
        g = golds(r)
        if not g:
            continue
        ans = r.get("answer") or ""
        raw_vals.append(anls(ans, g))
        strip_vals.append(anls(strip_cites(ans), g))
        if re.search(r"[A-Za-z]{4,}", ans) and not re.search(r"[가-힣]{2,}", ans):
            eng += 1
    print(f"Latest multimodal run: {name}")
    print(f"  harness overall ANLS/correctness: {overall}")
    print(f"  recomputed raw ANLS:           {sum(raw_vals)/len(raw_vals):.4f} (n={len(raw_vals)})")
    print(f"  after citation strip:          {sum(strip_vals)/len(strip_vals):.4f}")
    print(f"  English-only-ish answers:      {eng}/{len(rows)}")
    print("  sample answers:")
    for r in rows[:3]:
        print("   -", (r.get("answer") or "")[:120])

## 5. Inference practicality bench (needs GGUF)

This cell intentionally skips if weights are missing — it is **not** the QA harness.

In [ ]:
import sys
from pathlib import Path

gguf = Path("models/gemma-3-4b-it-q4_0.gguf")
if gguf.exists():
    !{sys.executable} scripts/bench_gemma_inference.py --repeats 2
else:
    print("Skip bench: models/gemma-3-4b-it-q4_0.gguf missing.")
    print("Fix: set DOWNLOAD_GGUF=True in section 1 and re-run that cell.")

## 6. Package all results for download

Creates `/content/pdf_vlm_all_results.zip` with eval runs, reports, figures, and inference bench outputs.

In [ ]:
import shutil
from pathlib import Path
from datetime import datetime

ROOT = Path("/content/pdf_vlm_repo")
export = Path("/content/pdf_vlm_export")
if export.exists():
    shutil.rmtree(export)
export.mkdir(parents=True)

# copy result trees (skip huge weights)
for rel in [
    "results/runs",
    "results/reports",
    "results/figures",
    "results/bench",
    "data/custom/colab_prepared.json",
]:
    src = ROOT / rel
    if not src.exists():
        print("skip missing", rel)
        continue
    dst = export / rel
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print("copied", rel)

# small index/manifest snapshots for reproducibility (not full faiss)
idx_meta = export / "indices_manifests"
idx_meta.mkdir(parents=True, exist_ok=True)
for man in Path(ROOT / "indices").glob("*/page_text/manifest.json"):
    doc = man.parents[1].name
    out = idx_meta / doc
    out.mkdir(exist_ok=True)
    shutil.copy2(man, out / "page_text_manifest.json")
    h = man.parents[1] / "hier_text" / "manifest.json"
    if h.exists():
        shutil.copy2(h, out / "hier_text_manifest.json")

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path = Path(f"/content/pdf_vlm_all_results_{stamp}")
shutil.make_archive(str(zip_path), "zip", export)
zip_file = Path(str(zip_path) + ".zip")
print("ZIP:", zip_file, "size_MB=", round(zip_file.stat().st_size / 1e6, 2))

# Colab download dialog
try:
    from google.colab import files
    files.download(str(zip_file))
except Exception as e:
    print("Auto-download failed (", e, "). Download manually from left file panel:", zip_file)